In [ ]:
import os
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from textblob import TextBlob
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from urllib.parse import unquote
from scipy.special import softmax
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import BartTokenizer, BartForConditionalGeneration

In [ ]:
load_dotenv()
api_token=os.getenv('HUGGINGFACEHUB_API_TOKEN')
os.environ['HF_TOKEN']=api_token
os.environ['HUGGINGFACEHUB_API_TOKEN']=api_token

In [ ]:
def get_stock_sector(stock_symbol):
    try:
        stock = yf.Ticker(stock_symbol)
        info = stock.info
        sector = info.get('sector', 'Sector information not available')
        industry=info.get('industry','Industry information not available')        
        return sector,industry
    except Exception as e:
        print("Error:", e)
        return None

In [ ]:
stock_symbol = input("Enter Stock Symbol: ")
sector,industry = get_stock_sector(stock_symbol)
if sector and industry:
    print("Stock:",stock_symbol)
    print("Sector:", sector)
    print('Industry:',industry)
elif sector:
    print("Stock:",stock_symbol)
    print("Sector:", sector)
    print('Industry:',"Falied to retrive industry information")
elif industry:
    print("Stock:",stock_symbol)
    print("Sector:", "Failed to retrieve sector information.")
    print('Industry:',industry)   
else:
    print("Failed to retrieve sector information.")

In [ ]:
def scraping_article(url):
    headers = {
    'User-Agent': 'Mozilla/5.0',
    }
    r=requests.get(url,headers=headers)
    soup=BeautifulSoup(r.text,'html.parser')
    paragraphs=soup.find_all('p')
    text= [paragraph.text for paragraph in paragraphs]
    words=' '.join(text).split(' ')
    article = ' '.join(words)
    return article

In [ ]:
def find_url(keyword):
    
    root = "https://www.google.com/"
    search_query = keyword.replace(" ", "+")
    link = f"https://www.google.com/search?q={search_query}&tbm=nws"
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(link, headers=headers)
    webpage = response.content
    soup = BeautifulSoup(webpage, 'html5lib')
    links = []
    for div_tag in soup.find_all('div', class_='Gx5Zad'):
        a_tag = div_tag.find('a')
        if a_tag:
            if 'href' in a_tag.attrs:
                href = a_tag['href']
                if href.startswith('/url?q='):
                    url = href.split('/url?q=')[1].split('&sa=')[0]
                    links.append(url)
    return links    

In [ ]:
def to_chunks(data):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=3000,
        chunk_overlap=50
    )
    docs=text_splitter.split_text(data)
    return docs

In [ ]:
def load_bart_model(model_name="facebook/bart-large-cnn"):
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name)
    return tokenizer, model

In [ ]:
def find_news_url(keyword, start_date, end_date):
    root = "https://www.google.com/"
    search_query = keyword.replace(" ", "+")
    link = f"{root}search?q={search_query}&tbm=nws&tbs=cdr:1,cd_min:{start_date},cd_max:{end_date}"

    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

    response = requests.get(link, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    news_links = []

    for article in soup.select('div.SoaBEf'):
        link = article.select_one('a')
        if link and 'href' in link.attrs:
            url = link['href']
            if url.startswith('/url?q='):
                url = unquote(url.split('/url?q=')[1].split('&sa=')[0])
            news_links.append(url)

    return news_links

In [ ]:
def summarize_text(tokenizer, model, text, max_chunk_length, summary_max_length):
    inputs = tokenizer(text, return_tensors="pt", max_length=max_chunk_length, truncation=True)
    summary_ids = model.generate(inputs["input_ids"], max_length=summary_max_length, min_length=200, length_penalty=2.0, num_beams=4, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
def summarize_article(url, model_name="facebook/bart-large-cnn"):
    data = scraping_article(url)
    chunks = to_chunks(data)
    # tokenizer, model=load_pegasus_model("google/pegasus-xsum")
    tokenizer, model = load_bart_model(model_name)
    summaries = []
    for chunk in chunks:
        chunk_text = chunk
        summary = summarize_text(tokenizer, model, chunk_text,3000,800)
        summaries.append(summary)
    concatenated_summaries = " ".join(summaries)
    #  Second summarization pass: Summarize the concatenated summaries
    intermediate_chunks = [concatenated_summaries[i:i+3000] for i in range(0, len(concatenated_summaries), 3000)]
    final_summaries = []
    for intermediate_chunk in intermediate_chunks:
        final_summary = summarize_text(tokenizer, model, intermediate_chunk,3000,800)
        final_summaries.append(final_summary)    
    final_summary_text = " ".join(final_summaries)    
    return final_summary_text

In [ ]:
url=find_url('elon musk')
# url="https://www.hrkatha.com/global-hr-news/tesla-to-lay-off-601-more-employees-amidst-market-challenges/"
# summary = summarize_article(url)
# print(summary)
print(len(url))

In [ ]:
def senti_model(model_name="cardiffnlp/twitter-roberta-base-sentiment"):
    tokenizer=AutoTokenizer.from_pretrained(model_name)
    model=AutoModelForSequenceClassification.from_pretrained(model_name)
    return tokenizer,model

In [ ]:
def find_senti(news_texts):
    tokenizer,model=senti_model()
    encoded=tokenizer(news_texts,return_tensors='pt',truncation=True, padding=True, max_length=512)
    output=model(**encoded)
    scores=output[0][0].detach().numpy()
    scores=softmax(scores)
    weights = {
        'neg': -1,
        'neu': 0,
        'pos': 1
    }
    probabilities = {
        'neg': scores[0],
        'neu': scores[1],
        'pos': scores[2]
    }
    compound_score = sum(probabilities[label] * weights[label] for label in probabilities)
    senti_dict={
        'neg':scores[0],
        'neu': scores[1],
        'pos': scores[2],
        'polarity':compound_score        
    }
    return senti_dict


In [ ]:
def subjectivity_analysis(text):
    blob = TextBlob(text)
    subjectivity = blob.sentiment.subjectivity
    return subjectivity

def relevance_scoring(summary, stock_name, domain_keywords):
    relevance_score = sum(1 for word in domain_keywords if word in summary) + (stock_name in summary)
    return relevance_score

def urgency_scoring(summary):
    # Look for time-sensitive words or phrases in the summary
    urgency_keywords = ['breaking', 'urgent', 'immediate']
    urgency_score = sum(1 for word in urgency_keywords if word in summary)
    return urgency_score

def mention_count(summary, stock_name, domain_keywords):
    # Count how many times the stock name and domain keywords appear in the summary
    count = sum(summary.count(word) for word in domain_keywords) + summary.count(stock_name)
    return count


In [ ]:
def senti_find(news_texts):
    from nltk.sentiment import SentimentIntensityAnalyzer
    sia=SentimentIntensityAnalyzer()
    senti =sia.polarity_scores(news_texts)
    return senti

In [ ]:
def extract_features(summary):
    # # Sentiment analysis
    sentiment_scores = find_senti(summary)
    subjectivity_score = subjectivity_analysis(summary)
    
    # # Financial analysis
    # relevance_score = relevance_scoring(summary, stock_name, domain_keywords)
    # urgency_score = urgency_scoring(summary)
    # mention_count_score = mention_count(summary, stock_name, domain_keywords)
    
    # Construct feature vector
    features = {
        'compound_sentiment_score': sentiment_scores['polarity'],
        'subjectivity_score': subjectivity_score,
        # 'relevance_score': relevance_score,
        # 'urgency_score': urgency_score,
        # 'mention_count': mention_count_score,
        'negative_sentiment_score': sentiment_scores['neg'],
        'neutral_sentiment_score': sentiment_scores['neu'],
        'positive_sentiment_score': sentiment_scores['pos']
    }
    
    return features
# def extract_features(summary):
#     # Sentiment analysis
#     sentiment_scores = senti_find(summary)
    
#     # Construct feature vector
#     features = {
#         'compound_sentiment_score': sentiment_scores['compound'],
#         'negative_sentiment_score': sentiment_scores['neg'],
#         'neutral_sentiment_score': sentiment_scores['neu'],
#         'positive_sentiment_score': sentiment_scores['pos']
#     }
    
#     return features


In [ ]:
scores=extract_features(summary,"tesla","technology")
print(scores)

In [ ]:
df=pd.read_csv("tesla_dataset.csv")
df.drop(df.columns[df.columns.str.contains('Unnamed',case = False)],axis = 1, inplace = True)
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df['Movement'] = (df['Close'] - df['Open'] > 0).astype(int)
df.head()

In [ ]:
df.value_counts("Movement")

In [ ]:
compound_sentiment_scores = []
negative_sentiment_scores = []
neutral_sentiment_scores = []
positive_sentiment_scores = []
subjectivity_score=[]

In [ ]:
for idx, row in df.iterrows():
    articles = [row[f'Article {i}'] for i in range(1, 6)]
    
    # Initialize sums for aggregation
    compound_sentiment_sum = 0
    negative_sentiment_sum = 0
    neutral_sentiment_sum = 0
    positive_sentiment_sum = 0
    subjectivity_sum=0
    
    # Process each article
    for article in articles:
        features = extract_features(article)
        compound_sentiment_sum += features['compound_sentiment_score']
        negative_sentiment_sum += features['negative_sentiment_score']
        neutral_sentiment_sum += features['neutral_sentiment_score']
        positive_sentiment_sum += features['positive_sentiment_score']
        subjectivity_sum+=features['subjectivity_score']
    
    # Calculate averages and append to lists
    num_articles = len(articles)
    compound_sentiment_scores.append(compound_sentiment_sum / num_articles)
    negative_sentiment_scores.append(negative_sentiment_sum / num_articles)
    neutral_sentiment_scores.append(neutral_sentiment_sum / num_articles)
    positive_sentiment_scores.append(positive_sentiment_sum / num_articles)
    subjectivity_score.append(subjectivity_sum/num_articles)

# Add aggregated features to DataFrame
df['avg_compound_sentiment_score'] = compound_sentiment_scores
df['avg_negative_sentiment_score'] = negative_sentiment_scores
df['avg_neutral_sentiment_score'] = neutral_sentiment_scores
df['avg_positive_sentiment_score'] = positive_sentiment_scores
df['avg_subjectivity_score']=subjectivity_score

# Display the DataFrame
print(df)

In [ ]:
df['Sentiment'] = df['avg_compound_sentiment_score'].apply(lambda x: 'Positive' if x >0 else 'Negative')
df.head()

In [ ]:
import pandas as pd


In [ ]:
df=pd.read_csv("update.csv")

In [ ]:
%pip install lightgbm

In [ ]:
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split, GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
    from sklearn.svm import SVC
    from sklearn.metrics import classification_report, accuracy_score
    from imblearn.over_sampling import SMOTE
    from sklearn.feature_selection import SelectKBest, f_classif

    # 1. Prepare the data
    features = ['avg_compound_sentiment_score', 'avg_negative_sentiment_score', 
                'avg_neutral_sentiment_score', 'avg_positive_sentiment_score', 
                'avg_subjectiivity_score']
    target = 'Movement'

    X = df[features].copy()
    y = df[target]

    # Feature engineering
    X['pos_neg_ratio'] = X['avg_positive_sentiment_score'] / (X['avg_negative_sentiment_score'] + 1e-10)
    X['compound_abs'] = np.abs(X['avg_compound_sentiment_score'])

    # Feature selection
    selector = SelectKBest(f_classif, k=5)
    X_new = selector.fit_transform(X, y)
    selected_features = X.columns[selector.get_support()].tolist()

    # 2. Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

    # 3. Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 4. Handle class imbalance
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

    # 5. Define models and parameters for grid search
    models = {
        'RandomForest': (RandomForestClassifier(random_state=42),
                        {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}),
        'GradientBoosting': (GradientBoostingClassifier(random_state=42),
                            {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.1]}),
        'SVM': (SVC(random_state=42, probability=True),
                {'C': [0.1, 1, 10], 'kernel': ['rbf', 'linear']})
    }

    # 6. Perform grid search and train models
    best_models = {}
    for name, (model, params) in models.items():
        grid_search = GridSearchCV(model, params, cv=5, n_jobs=-1)
        grid_search.fit(X_train_resampled, y_train_resampled)
        best_models[name] = grid_search.best_estimator_

    # 7. Make predictions and evaluate
    for name, model in best_models.items():
        y_pred = model.predict(X_test_scaled)
        print(f"\n{name} Results:")
        print(classification_report(y_test, y_pred))
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")

    # 8. Ensemble the best models
    from sklearn.ensemble import VotingClassifier

    ensemble = VotingClassifier(
        estimators=[(name, model) for name, model in best_models.items()],
        voting='soft'
    )
    ensemble.fit(X_train_resampled, y_train_resampled)
    y_pred_ensemble = ensemble.predict(X_test_scaled)

    print("\nEnsemble Model Results:")
    print(classification_report(y_test, y_pred_ensemble))
    print(f"Accuracy: {accuracy_score(y_test, y_pred_ensemble):.2f}")

    # Print feature importances for RandomForest
    rf_model = best_models['RandomForest']
    importances = rf_model.feature_importances_
    for feature, importance in zip(selected_features, importances):
        print(f"{feature}: {importance:.4f}")

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(10, 6))

# Group by Sentiment and Movement and count occurrences
plot_data = df.groupby(['Sentiment', 'Movement']).size().unstack(fill_value=0)

# Plotting the grouped data
plot_data.plot(kind='bar', stacked=True)
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.title('Stock Movement Based on Sentiment')
plt.xticks(rotation=0)
plt.legend(title='Movement', labels=['Movement = 0', 'Movement = 1'])
plt.tight_layout()

plt.show()

In [ ]:
df_test=df.copy()

In [ ]:
# df.to_csv("update.csv")

In [ ]:
df_test.drop(["Sentiment"],axis=1,inplace=True)

In [ ]:
def categorize_sentiment(score):
    if score > 0:
        return 'Positive'
    elif score <= 0:
        return 'Negative'
    else:
        return 'Neutral'

df_test['Sentiment'] = df_test['avg_compound_sentiment_score'].apply(categorize_sentiment)

In [ ]:
plt.figure(figsize=(10, 6))

# Group by Sentiment and Movement and count occurrences
plot_data = df_test.groupby(['Sentiment', 'Movement']).size().unstack(fill_value=0)

# Plotting the grouped data
plot_data.plot(kind='bar', stacked=True)
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.title('Stock Movement Based on Sentiment')
plt.xticks(rotation=0)
plt.legend(title='Movement', labels=['Movement = 0', 'Movement = 1'])
plt.tight_layout()

plt.show()

In [ ]:
df.to_csv("no_neutral.csv")